# Day 62 — Month 3 Capstone
**GlobalMart 2023–2024 Full Business Analytics Pipeline**

---

> **Scenario:**
>
> GlobalMart operates across 5 regions and 6 product categories, serving B2C, B2B, and Enterprise segments.
> You have been handed 1,000 orders spanning two full years (2023–2024) and asked to build
> a complete analytics pipeline from raw data to boardroom report — in one notebook.
>
> This capstone integrates everything from Month 3:
> **Memory Optimization · Feature Engineering · NumPy Analytics · Time Series ·
> MultiIndex GroupBy · Advanced Aggregations · Chunked Pipeline · Executive Reporting**
>
> No hints. No skeletons beyond task descriptions. Build it like a client is watching.

---
**Score: 120 pts + 15★ Bonus** | Dataset: 1,000 orders | seed=62 | 2023-01-01 to 2024-12-31


## Section 1 — Raw Data
*Run once. Never modify. Work on `df` throughout.*

In [31]:
import numpy as np
import pandas as pd
import warnings, time
warnings.filterwarnings('ignore')

rng = np.random.default_rng(seed=62)
n   = 1000

raw = pd.DataFrame({
    'order_id'     : [f'GM{10000+i}' for i in range(n)],
    'customer_id'  : [f'C{rng.integers(1,201):03d}' for _ in range(n)],
    'order_date'   : pd.to_datetime(
                         rng.choice(pd.date_range('2023-01-01','2024-12-31'), n, replace=True)),
    'region'       : rng.choice(['North','South','East','West','Central'], n,
                                 p=[0.22,0.20,0.20,0.20,0.18]),
    'category'     : rng.choice(['Electronics','Clothing','Home','Sports','Books','Beauty'], n,
                                 p=[0.25,0.22,0.18,0.15,0.10,0.10]),
    'segment'      : rng.choice(['B2C','B2B','Enterprise'], n, p=[0.55,0.30,0.15]),
    'status'       : rng.choice(['Delivered','Returned','Cancelled','Pending'], n,
                                 p=[0.68,0.18,0.08,0.06]),
    'units'        : rng.integers(1, 51, n).astype('int64'),
    'unit_price'   : rng.uniform(20, 800, n).astype('float64'),
    'discount_pct' : rng.choice([0,5,10,15,20,25], n).astype('int64'),
    'shipping_cost': rng.uniform(10, 200, n).astype('float64'),
    'rating'       : rng.choice([1,2,3,4,5], n, p=[0.05,0.10,0.20,0.35,0.30]).astype('int64'),
})
raw['revenue'] = (raw['units'] * raw['unit_price'] * (1 - raw['discount_pct']/100)).round(2)
raw['profit']  = (raw['revenue'] * rng.uniform(0.05, 0.35, n)).round(2)

df = raw.copy()
print(f"GlobalMart Dataset: {df.shape[0]} orders | {df['order_date'].dt.year.min()}–{df['order_date'].dt.year.max()}")
print(f"Regions: {sorted(df['region'].unique())} | Categories: {sorted(df['category'].unique())}")
print(f"Total Revenue: ₹{df['revenue'].sum():>15,.2f}")
print(f"Total Profit:  ₹{df['profit'].sum():>15,.2f}")
df.head(3)


GlobalMart Dataset: 1000 orders | 2023–2024
Regions: ['Central', 'East', 'North', 'South', 'West'] | Categories: ['Beauty', 'Books', 'Clothing', 'Electronics', 'Home', 'Sports']
Total Revenue: ₹   9,353,311.11
Total Profit:  ₹   1,861,844.29


,order_id,customer_id,order_date,region,category,segment,status,units,unit_price,discount_pct,shipping_cost,rating,revenue,profit
0,GM10000,C090,2023-07-02,Central,Electronics,Enterprise,Delivered,12,121.996333,15,186.599525,3,1244.36,239.49
1,GM10001,C122,2023-10-07,North,Sports,Enterprise,Cancelled,12,267.981166,15,61.704875,3,2733.41,219.33
2,GM10002,C114,2023-11-16,Central,Clothing,B2B,Delivered,16,79.355068,25,94.835526,4,952.26,113.52


## Task A — Memory Optimization (15 pts)

The pipeline will run on GlobalMart's 10 GB production file. Optimise memory before any analysis.

**A1 (5 pts):** Compute baseline memory using `memory_usage(deep=True)`. Print per-column KB.
Store total in `baseline_kb`.

**A2 (10 pts):** Create `df_opt = df.copy()`. Apply all safe downcasts:

| Column | → dtype | Reason |
|--------|---------|--------|
| `units` | int8 | max 50 fits int8 |
| `discount_pct` | int8 | values 0–25 |
| `rating` | int8 | values 1–5 |
| `unit_price` | float32 | display precision OK |
| `shipping_cost` | float32 | display precision OK |
| `region`, `category`, `segment`, `status` | category | low cardinality |

**Do NOT downcast `revenue` or `profit`** — both are aggregated in Task F.
Print optimised KB, % reduction, and an NRA insight.


In [32]:
# Task A — Memory Optimization
# A1: baseline audit
mem_usage = df.memory_usage(deep=True)
print("\n--- A1: Baseline Memory Audit (KB) ---")
for col in mem_usage.index:
    print(f"{col:20s} {mem_usage[col]/1024:8.2f} KB")
baseline_kb = mem_usage.sum()/1024
print(f"Total baseline: {baseline_kb:.2f} KB")

# A2: dtype optimisation — df_opt = df.copy(), apply table above
# revenue and profit stay float64
# Print: Optimised: XX.XX KB | Reduction: XX.X%
# NRA insight

df_opt = df.copy()
for col, dt in [('units','int8'), ('discount_pct','int8'), ('rating','int8'),
                ('unit_price','float32'), ('shipping_cost','float32')]:
    df_opt[col] = df_opt[col].astype(dt)
for col in ['region','category','segment','status']:
    df_opt[col] = df_opt[col].astype('category')
# revenue and profit remain float64

opt_kb = df_opt.memory_usage(deep=True).sum()/1024
reduction = (1 - opt_kb/baseline_kb) * 100
print(f"\nA2: Optimised: {opt_kb:.2f} KB | Reduction: {reduction:.1f}%")
print("NRA Insight: By downcasting numeric columns and converting low‑cardinality strings to category,")
print("             we cut memory by over 60%, preparing the pipeline for a 10 GB production file.")




--- A1: Baseline Memory Audit (KB) ---
Index                    0.13 KB
order_id                62.50 KB
customer_id             59.57 KB
order_date               7.81 KB
region                  60.45 KB
category                62.77 KB
segment                 59.61 KB
status                  64.16 KB
units                    7.81 KB
unit_price               7.81 KB
discount_pct             7.81 KB
shipping_cost            7.81 KB
rating                   7.81 KB
revenue                  7.81 KB
profit                   7.81 KB
Total baseline: 431.69 KB

A2: Optimised: 162.01 KB | Reduction: 62.5%
NRA Insight: By downcasting numeric columns and converting low‑cardinality strings to category,
             we cut memory by over 60%, preparing the pipeline for a 10 GB production file.


## Task B — Feature Engineering (20 pts)

Build all derived columns on `df` (the original copy — not df_opt). Every column below is used in Tasks C–F.

**B1 (8 pts):** Date features — add these columns:
- `year` → `dt.year`
- `month` → `dt.to_period('M')`
- `quarter` → `dt.to_period('Q')`
- `day_of_week` → `dt.day_name()`
- `is_weekend` → True if Saturday or Sunday (`dt.dayofweek >= 5`)

**B2 (6 pts):** Financial features:
- `net_revenue` = revenue − shipping_cost (round 2 dp)
- `profit_margin` = (profit / revenue × 100) (round 2 dp)
- `rev_tier` — classify each order using **one nested `np.where()`**:
  - `'High'` if revenue > 66th percentile
  - `'Medium'` if revenue > 33rd percentile
  - `'Low'` otherwise

Use `np.percentile(df['revenue'].values, 33)` and `np.percentile(df['revenue'].values, 66)`.

**B3 (6 pts):** Status flags:
- `is_returned` = 1 if status == 'Returned' else 0 (int8)
- `is_cancelled` = 1 if status == 'Cancelled' else 0 (int8)

Print `df.shape` after all additions. Print `df[['net_revenue','profit_margin','rev_tier','is_returned','is_cancelled']].describe().round(2)`.


In [33]:
# Task B — Feature Engineering
# B1: date features (year, month, quarter, day_of_week, is_weekend)
df['year']         = df['order_date'].dt.year
df['month']        = df['order_date'].dt.to_period('M')
df['quarter']      = df['order_date'].dt.to_period('Q')
df['day_of_week']  = df['order_date'].dt.day_name()
df['is_weekend']   = df['order_date'].dt.dayofweek >= 5

# B2: financial features (net_revenue, profit_margin, rev_tier via np.where)
df['net_revenue']  = (df['revenue'] - df['shipping_cost']).round(2)
df['profit_margin']= (df['profit'] / df['revenue'] * 100).round(2)
rev_p33 = np.percentile(df['revenue'].values, 33)
rev_p66 = np.percentile(df['revenue'].values, 66)
df['rev_tier']     = np.where(df['revenue'] > rev_p66, 'High',
                     np.where(df['revenue'] > rev_p33, 'Medium', 'Low'))

# B3: status flags (is_returned, is_cancelled as int8)

# Print df.shape and describe() of new columns
df['is_returned']  = (df['status'] == 'Returned').astype('int8')
df['is_cancelled'] = (df['status'] == 'Cancelled').astype('int8')

print(f"\nB shape: {df.shape} | weekend orders: {df['is_weekend'].sum()}")
print("B rev_tier distribution:")
print(df['rev_tier'].value_counts().to_dict())
print("\nB Descriptive statistics for new financial columns:")
print(df[['net_revenue','profit_margin','rev_tier','is_returned','is_cancelled']].describe().round(2))



B shape: (1000, 24) | weekend orders: 307
B rev_tier distribution:
{'High': 340, 'Low': 330, 'Medium': 330}

B Descriptive statistics for new financial columns:
       net_revenue  profit_margin  is_returned  is_cancelled
count      1000.00        1000.00      1000.00       1000.00
mean       9246.85          20.20         0.17          0.08
std        7718.82           8.89         0.38          0.27
min        -142.57           5.02         0.00          0.00
25%        2648.08          12.31         0.00          0.00
50%        7513.67          20.34         0.00          0.00
75%       13795.56          28.20         0.00          0.00
max       36476.40          34.97         1.00          1.00


## Task C — NumPy Analytics (15 pts)

Use NumPy arrays directly — no Pandas `describe()` or `corr()`.

**C1 (5 pts):** Compute for `revenue` AND `profit` arrays separately:
mean, median, std, min, max, p25, p75, IQR. Format all as ₹X,XXX.XX. Print as a clean comparison table.

**C2 (4 pts):** Compute the **Pearson correlation** between revenue and profit using `np.corrcoef()`.
Print the coefficient rounded to 4 dp. Interpret in one sentence: what does this correlation mean
for the business's pricing reliability?

**C3 (3 pts):** Compute the **weighted average discount** where the weight is `units` sold
(heavier orders count more). Use `np.average(disc, weights=units)`.
Compare with the simple mean discount. Write one sentence on why weighted avg matters here.

**C4 (3 pts):** Identify revenue outliers: orders where the z-score of revenue exceeds ±2.
Print the count. Print those outliers' `order_id`, `category`, `region`, and `revenue`.


In [34]:
# Task C — NumPy Analytics
# C1: revenue and profit stats table
rev  = df['revenue'].values
prof = df['profit'].values

def stats_dict(arr):
    return {
        'mean':   np.mean(arr),
        'median': np.median(arr),
        'std':    np.std(arr),
        'min':    np.min(arr),
        'max':    np.max(arr),
        'p25':    np.percentile(arr, 25),
        'p75':    np.percentile(arr, 75),
        'IQR':    np.percentile(arr, 75) - np.percentile(arr, 25)
    }

rev_stats = stats_dict(rev)
prof_stats = stats_dict(prof)

print("\n--- C1: Revenue vs Profit Comparison ---")
print(f"{'Statistic':>8}  {'Revenue (₹)':>18}  {'Profit (₹)':>18}")
print(f"{'':->8}  {'':->18}  {'':->18}")
for k in ['mean','median','std','min','max','p25','p75','IQR']:
    print(f"{k:>8}  ₹{rev_stats[k]:>16,.2f}  ₹{prof_stats[k]:>16,.2f}")

# C2: Pearson correlation + interpretation
corr = np.corrcoef(rev, prof)[0,1]
print(f"\nC2: Pearson correlation = {corr:.4f}")
print("Interpretation: A strong positive correlation (0.8223) indicates that higher revenue orders")
print("                 reliably translate into higher profit, reinforcing consistent pricing integrity.")

# C3: weighted avg discount vs simple mean + explanation
w_disc = np.average(df['discount_pct'].values, weights=df['units'].values)
simple_mean = df['discount_pct'].mean()
print(f"\nC3: Weighted avg discount = {w_disc:.2f}% | Simple mean = {simple_mean:.2f}%")
print("Why weighted matters: Larger orders (more units) have a bigger impact on revenue; weighting")
print("                      by units gives a truer picture of the effective discount applied to overall sales volume.")

# C4: z-score outliers (|z| > 2)
z = (rev - np.mean(rev)) / np.std(rev)
outliers = df[np.abs(z) > 2][['order_id','category','region','revenue']]
print(f"\nC4: {len(outliers)} revenue outliers (|z|>2):")
print(outliers.head(5).to_string(index=False))




--- C1: Revenue vs Profit Comparison ---
Statistic         Revenue (₹)          Profit (₹)
--------  ------------------  ------------------
    mean  ₹        9,353.31  ₹        1,861.84
  median  ₹        7,628.26  ₹        1,226.65
     std  ₹        7,714.29  ₹        1,891.48
     min  ₹           29.34  ₹            2.92
     max  ₹       36,536.41  ₹       10,451.90
     p25  ₹        2,774.09  ₹          514.27
     p75  ₹       13,956.61  ₹        2,593.71
     IQR  ₹       11,182.52  ₹        2,079.44

C2: Pearson correlation = 0.8223
Interpretation: A strong positive correlation (0.8223) indicates that higher revenue orders
                 reliably translate into higher profit, reinforcing consistent pricing integrity.

C3: Weighted avg discount = 12.46% | Simple mean = 12.70%
Why weighted matters: Larger orders (more units) have a bigger impact on revenue; weighting
                      by units gives a truer picture of the effective discount applied to overall sales volu

## Task D — Time Series Analysis (20 pts)

**D1 (8 pts):** Set `order_date` as index on a copy (`df_ts`). Resample to **monthly** frequency.
Aggregate simultaneously:
- `monthly_rev` = sum of revenue
- `monthly_profit` = sum of profit
- `monthly_units` = sum of units

Then add:
- `profit_margin_pct` = monthly_profit / monthly_rev × 100 (round 2 dp)
- `mom_growth` = month-over-month % change on monthly_rev (round 2 dp)
- `rolling_3m` = 3-month rolling average of monthly_rev (round 2 dp)

Print the full 24-month table.

**D2 (6 pts):** Identify:
- Best and worst month by revenue (print formatted)
- The month with the highest profit margin %
- The longest consecutive streak of positive MoM growth (hint: use a loop on the `mom_growth` series)

Write NRA insight on the worst month.

**D3 (6 pts):** Year-over-year comparison.
Group `df` by `year` and compute total revenue, total profit, total orders, avg order value.
Print a clean YoY table. Compute YoY revenue growth % and print.
Write NRA insight: is GlobalMart growing? What should leadership do?


In [35]:
# Task D — Time Series
# D1: monthly resample with multi-column agg + derived metrics
df_ts = df.set_index('order_date').sort_index()
monthly = df_ts[['revenue','profit','units']].resample('ME').agg(
    monthly_rev    = ('revenue','sum'),
    monthly_profit = ('profit','sum'),
    monthly_units  = ('units','sum')
).round(2)

monthly['profit_margin_pct'] = (monthly['monthly_profit'] / monthly['monthly_rev'] * 100).round(2)
monthly['mom_growth']        = monthly['monthly_rev'].pct_change().mul(100).round(2)
monthly['rolling_3m']        = monthly['monthly_rev'].rolling(3).mean().round(2)

print("\n--- D1: Full Monthly Table ---")
print(monthly.to_string())

# D2: best/worst month, highest margin month, longest +MoM streak
#     NRA on worst month
best_m   = monthly['monthly_rev'].idxmax()
worst_m  = monthly['monthly_rev'].idxmin()
hm_margin= monthly['profit_margin_pct'].idxmax()

streak = cur = 0
for v in monthly['mom_growth'].dropna():
    if v > 0:
        cur += 1
        streak = max(streak, cur)
    else:
        cur = 0

print(f"\nD2 Best month:        {best_m.strftime('%B %Y')} — ₹{monthly.loc[best_m, 'monthly_rev']:,.2f}")
print(f"D2 Worst month:       {worst_m.strftime('%B %Y')} — ₹{monthly.loc[worst_m, 'monthly_rev']:,.2f}")
print(f"D2 Highest margin:    {hm_margin.strftime('%B %Y')} — {monthly.loc[hm_margin, 'profit_margin_pct']}%")
print(f"D2 Longest +MoM streak: {streak} months")
print("NRA on worst month: May 2024 had the lowest revenue, likely due to seasonal lull or fewer high‑value orders.")
print("                    Consider targeted campaigns during low‑performing months to smooth cash flow.")

# D3: YoY comparison table + growth % + NRA
yoy = df.groupby('year').agg(
    rev        = ('revenue','sum'),
    profit     = ('profit','sum'),
    orders     = ('order_id','count'),
    avg_order  = ('revenue','mean')
).round(2)
yoy['yoy_growth'] = yoy['rev'].pct_change().mul(100).round(1)

print("\n--- D3: YoY Comparison ---")
print(yoy.to_string())
print(f"\nYoY Revenue Growth: +{yoy.loc[2024,'yoy_growth']:.1f}%")
print("NRA: GlobalMart shows minimal growth (0.9%). Leadership should focus on customer acquisition,")
print("     increasing average order value, and expanding in high‑performing region‑category pairs.")


--- D1: Full Monthly Table ---
            monthly_rev  monthly_profit  monthly_units  profit_margin_pct  mom_growth  rolling_3m
order_date                                                                                       
2023-01-31    359062.92        82879.53           1226              23.08         NaN         NaN
2023-02-28    603896.19       114155.63           1516              18.90       68.19         NaN
2023-03-31    418877.04        72153.11           1126              17.23      -30.64   460612.05
2023-04-30    295745.50        58534.60            824              19.79      -29.40   439506.24
2023-05-31    554552.61       112946.04           1372              20.37       87.51   423058.38
2023-06-30    338477.58        69879.44            948              20.65      -38.96   396258.56
2023-07-31    343301.71        65262.83           1118              19.01        1.43   412110.63
2023-08-31    419768.50        80397.93           1088              19.15       22.27 

## Task E — MultiIndex & Advanced GroupBy (30 pts)

**E1 (10 pts):** Named aggregation across `['region','category']` — compute in **one** `.agg()` call:
- `total_rev` = sum of revenue
- `total_profit` = sum of profit
- `avg_order` = mean of revenue
- `order_count` = count of orders

Round to 2 dp. Print the full 30-row MultiIndex table.
Unstack `category` to create a `total_rev` cross-tab (5 regions × 6 categories).
Print the cross-tab. Identify and print the best AND worst region-category pair.
Write NRA insight.

**E2 (6 pts):** Using `transform`:
- Add `seg_total` = total revenue per segment for each order's segment
- Add `seg_share_pct` = each order's % of its segment total (3 dp)

Print the top 5 orders by `seg_share_pct` with columns `['order_id','segment','revenue','seg_share_pct']`.

**E3 (6 pts):** Category-level health report using a single `.agg()`:
- `total_orders`, `returned` (sum of is_returned), `cancelled` (sum of is_cancelled), `avg_rating` (mean of rating)

Add a computed column `return_rate_pct` = returned / total_orders × 100 (round 1 dp).
Print the full table sorted by `return_rate_pct` descending.
Write NRA: which category needs urgent quality attention?

**E4 (4 pts):** Pivot table — segment (rows) × year (columns) — values = total revenue.
Print. Which segment grew the most from 2023 to 2024? Write one line.

**E5 (4 pts):** Customer loyalty using `cumcount`:
Sort by `customer_id` then `order_date`. Add `order_seq` (1-indexed per customer).
Print: how many orders are repeat (seq ≥ 2)? Print the top customer (most orders).
Write NRA: what does the repeat rate tell GlobalMart's retention team?


In [36]:
# Task E — MultiIndex & Advanced GroupBy
# E1: named agg (30 rows), unstack cross-tab, best+worst pair, NRA
summary = df.groupby(['region','category']).agg(
    total_rev   = ('revenue','sum'),
    total_profit= ('profit','sum'),
    avg_order   = ('revenue','mean'),
    order_count = ('revenue','count')
).round(2)

print("\n--- E1: Full 30‑row MultiIndex Table ---")
print(summary)

cross_tab = summary['total_rev'].unstack()
print("\nCross‑tab (region × category):")
print(cross_tab.to_string())

best_pair  = summary['total_rev'].idxmax()
worst_pair = summary['total_rev'].idxmin()
print(f"\nBest region‑category:  {best_pair[0]} × {best_pair[1]} — ₹{summary.loc[best_pair,'total_rev']:,.2f}")
print(f"Worst region‑category: {worst_pair[0]} × {worst_pair[1]} — ₹{summary.loc[worst_pair,'total_rev']:,.2f}")
print("NRA: South × Electronics dominates revenue, suggesting strong demand and effective distribution.")
print("     Central × Beauty is weakest; limited product variety or lower purchasing power may be factors.")

# E2: transform for segment share %
df['seg_total']     = df.groupby('segment')['revenue'].transform('sum')
df['seg_share_pct'] = (df['revenue'] / df['seg_total'] * 100).round(3)

print("\n--- E2: Top 5 orders by segment share ---")
top5_share = df.nlargest(5, 'seg_share_pct')[['order_id','segment','revenue','seg_share_pct']]
print(top5_share.to_string(index=False))

# E3: category health report (return_rate, avg_rating), NRA
cat_health = df.groupby('category').agg(
    total_orders = ('order_id','count'),
    returned     = ('is_returned','sum'),
    cancelled    = ('is_cancelled','sum'),
    avg_rating   = ('rating','mean')
).round(2)
cat_health['return_rate_pct'] = (cat_health['returned'] / cat_health['total_orders'] * 100).round(1)
cat_health = cat_health.sort_values('return_rate_pct', ascending=False)

print("\n--- E3: Category Health (sorted by return rate) ---")
print(cat_health.to_string())
print("NRA: Books has the highest return rate (24.8%). Poor descriptions, damaged items, or mismatched expectations")
print("     could be the cause. Urgent quality audits and better product detail pages are recommended.")

# E4: pivot segment × year
pivot = df.pivot_table(values='revenue', index='segment', columns='year', aggfunc='sum').round(2)
print("\n--- E4: Segment × Year Pivot ---")
print(pivot.to_string())
# Identify fastest growing segment
growth = pivot.pct_change(axis=1).iloc[:,-1] * 100
print(f"\nSegment growth rates: {growth.to_dict()}")
fastest = growth.idxmax()
print(f"Fastest growing segment: {fastest}")

# E5: customer loyalty via cumcount, NRA
df_sorted = df.sort_values(['customer_id','order_date']).copy()
df_sorted['order_seq'] = df_sorted.groupby('customer_id').cumcount() + 1
repeat = (df_sorted['order_seq'] >= 2).sum()
top_cust = df_sorted.loc[df_sorted['order_seq'].idxmax(), ['customer_id','order_seq']]

print(f"\n--- E5: Customer Loyalty ---")
print(f"Repeat order rows: {repeat}")
print(f"Top customer: {top_cust['customer_id']} with {top_cust['order_seq']} orders")
print("NRA: 80% of rows are repeat orders, indicating strong customer retention. The loyalty team")
print("     should continue nurturing high‑frequency customers with personalised offers.")



--- E1: Full 30‑row MultiIndex Table ---
                     total_rev  total_profit  avg_order  order_count
region  category                                                    
Central Beauty        83272.79      14451.60    5948.06           14
        Books        103761.60      21838.40    6103.62           17
        Clothing     286418.55      59297.55    8424.07           34
        Electronics  502254.24      90068.45   10463.63           48
        Home         323440.95      68382.29   11551.46           28
        Sports       252842.98      54617.10    9724.73           26
East    Beauty       184953.34      34287.13    8406.97           22
        Books        280185.77      66420.40   11674.41           24
        Clothing     313643.38      61407.27   10117.53           31
        Electronics  502547.32      94560.12   10924.94           46
        Home         428513.99      87096.57    7935.44           54
        Sports       418268.21      94509.53    9958.77      

## Task F — Production Pipeline + Executive Report (20 pts)

**F1 (12 pts):** Save `df` to `/tmp/orders_day62.csv` (no index).
Build a chunked pipeline (`chunksize=200`, 5 chunks) that produces **quarter × region** totals
for both `revenue` and `profit` simultaneously.

Requirements inside the loop:
- `usecols=['order_date','region','category','revenue','profit']`
- Convert `region` and `category` to `category` dtype
- Keep `revenue` and `profit` as float64
- Add `quarter` using `dt.to_period('Q')`
- GroupBy `['quarter','region']` and sum both columns

After the loop: concat + `groupby(level=[0,1]).sum().round(2)`.
Print the full result. Print `Match full-load: True/False`.

**F2 (8 pts):** Executive Summary — print exactly these 8 lines using variables computed above:
```
═══════════════════════════════════════════════════
   GlobalMart 2023–2024 Executive Analytics Report
═══════════════════════════════════════════════════
Total Revenue:        ₹X,XXX,XXX.XX
Total Profit:         ₹X,XXX,XXX.XX
Overall Profit Margin: XX.X%
Best Month:           [Month YYYY] — ₹X,XXX,XXX.XX
Worst Month:          [Month YYYY] — ₹X,XXX,XXX.XX
Top Region-Category:  [Region] × [Category] — ₹XXX,XXX.XX
YoY Revenue Growth:   +X.X%
Memory Saved:         XX.X% (XXX.XX KB → XXX.XX KB)
═══════════════════════════════════════════════════
```


In [37]:
# Task F — Production Pipeline + Executive Report
# F1: chunked quarter × region pipeline (chunksize=200)
#     usecols, category dtype, revenue/profit float64
#     concat + groupby level=[0,1] sum
#     Match full-load verification
df.to_csv('orders_day62.csv', index=False)

chunks = []
for chunk in pd.read_csv('orders_day62.csv', chunksize=200, parse_dates=['order_date'],
                         usecols=['order_date','region','category','revenue','profit']):
    chunk['region']   = chunk['region'].astype('category')
    chunk['category'] = chunk['category'].astype('category')
    chunk['quarter']  = chunk['order_date'].dt.to_period('Q')
    chunk_grouped = chunk.groupby(['quarter','region'])[['revenue','profit']].sum()
    chunks.append(chunk_grouped)

pipeline_result = pd.concat(chunks).groupby(level=[0,1]).sum().round(2)

# Build full‑load check with categorical dtype to match pipeline
full_check = (df.assign(region=df['region'].astype('category'),
                        category=df['category'].astype('category'))
                .groupby(['quarter','region'])[['revenue','profit']]
                .sum().round(2))

print("\n--- F1: Chunked Pipeline Result ---")
print(pipeline_result.to_string())
print(f"\nMatch full‑load: {pipeline_result.equals(full_check)} | shape: {pipeline_result.shape}")

# F2: Executive Summary — all 8 lines using computed variables
overall_margin = (df['profit'].sum() / df['revenue'].sum() * 100)

print(f"\n{'═'*51}")
print(f"   GlobalMart 2023–2024 Executive Analytics Report")
print(f"{'═'*51}")
print(f"Total Revenue:         ₹{df['revenue'].sum():>14,.2f}")
print(f"Total Profit:          ₹{df['profit'].sum():>14,.2f}")
print(f"Overall Profit Margin: {overall_margin:>10.1f}%")
print(f"Best Month:            {best_m.strftime('%B %Y')} — ₹{monthly.loc[best_m,'monthly_rev']:,.2f}")
print(f"Worst Month:           {worst_m.strftime('%B %Y')} — ₹{monthly.loc[worst_m,'monthly_rev']:,.2f}")
print(f"Top Region-Category:   {best_pair[0]} × {best_pair[1]} — ₹{summary.loc[best_pair,'total_rev']:,.2f}")
print(f"YoY Revenue Growth:    +{yoy.loc[2024,'yoy_growth']:.1f}%")
print(f"Memory Saved:          {reduction:.1f}% ({baseline_kb:.2f} KB → {opt_kb:.2f} KB)")
print(f"{'═'*51}")



--- F1: Chunked Pipeline Result ---
                   revenue    profit
quarter region                      
2023Q1  Central  239470.86  36274.80
        East     328282.40  73360.05
        North    340640.19  64198.77
        South    220760.80  46895.71
        West     252681.90  48458.94
2023Q2  Central  196201.73  40220.52
        East     307185.77  69955.46
        North    263274.13  52176.79
        South    126436.23  24676.83
        West     295677.83  54330.48
2023Q3  Central  232938.20  51153.05
        East     204789.02  43158.02
        North    219359.46  35436.43
        South    295443.85  64577.52
        West     162583.16  21149.22
2023Q4  Central  137456.16  30709.66
        East     185418.68  41443.02
        North    213033.90  43548.45
        South    197910.90  38713.57
        West     235176.11  44660.96
2024Q1  Central  201484.63  40392.17
        East     300845.34  61471.51
        North    206166.82  36852.56
        South    274853.90  67129.09
 

## Section 4 — Scoring Rubric

| Task | Sub | Description | Pts |
|------|-----|-------------|-----|
| A | 1 | Per-column KB printed; `baseline_kb` stored (~384 KB) | 5 |
| A | 2 | 9 downcasts applied; revenue+profit left float64; NRA | 10 |
| B | 1 | 5 date features added correctly | 8 |
| B | 2 | net_revenue, profit_margin, rev_tier (nested np.where) | 6 |
| B | 3 | is_returned, is_cancelled as int8; df.shape + describe printed | 6 |
| C | 1 | 8 stats each for revenue AND profit; ₹ formatted | 5 |
| C | 2 | np.corrcoef used; coefficient correct (0.8223); interpreted | 4 |
| C | 3 | np.average with weights; compared to simple mean; explained | 3 |
| C | 4 | z-score formula correct; 49 outliers; outlier rows printed | 3 |
| D | 1 | resample ME; 3-col agg; profit_margin_pct; mom_growth; rolling_3m; 24-row table | 8 |
| D | 2 | Best/worst month correct; highest margin month; streak count; NRA | 6 |
| D | 3 | YoY table correct (2023 vs 2024); growth %; NRA | 6 |
| E | 1 | Named agg 30 rows; unstack cross-tab; best+worst pair; NRA | 10 |
| E | 2 | transform (no merge); seg_share_pct; top 5 printed | 6 |
| E | 3 | Category health table; return_rate_pct; sorted; NRA | 6 |
| E | 4 | Pivot segment × year; fastest-growing segment | 4 |
| E | 5 | cumcount+1; repeat count (801); top customer C062 (12); NRA | 4 |
| F | 1 | usecols, chunksize=200, category dtype, float64 agg, Match=True | 12 |
| F | 2 | All 8 executive lines correct and formatted | 8 |
| **Total** | | | **120** |

---

### ★ Bonus (15 pts)

**Bonus 1 — Weekend vs Weekday Analysis (5 pts)**
Using `is_weekend`, compare avg revenue, avg profit_margin, and order count between weekend and weekday orders.
Present as a clean pivot table. Write NRA: should GlobalMart run weekend promotions?

**Bonus 2 — Cancellation Risk Score (5 pts)**
Build a category-level risk score:
`risk_score = return_rate_pct × 0.6 + (cancelled/total_orders×100) × 0.4`
Print categories ranked by risk_score descending. Write NRA identifying the highest-risk category and recommended action.

**Bonus 3 — Customer Segmentation Summary (5 pts)**
For each `segment` (B2C, B2B, Enterprise), compute:
avg order value, avg profit margin, return rate, repeat order rate (rows with order_seq ≥ 2 / total rows).
Print a clean 3-row summary. Write NRA: which segment is most valuable and which needs most attention?

---

### Key Numbers to Hit

| Metric | Expected |
|--------|----------|
| A baseline | ~384 KB |
| A optimised | ~145 KB |
| A reduction | ~62% |
| C correlation | 0.8223 |
| C weighted discount | 12.46% |
| C outliers | 49 |
| D best month | February 2023 |
| D worst month | May 2024 |
| E1 shape | (30, 4) |
| E1 best pair | South × Electronics |
| E3 highest return rate | Books (24.8%) |
| E5 repeat rows | 801 |
| E5 top customer | C062 (12 orders) |
| F Match | True |
| F pipeline shape | (40, 2) |

---

### Interview Frame

> "A capstone pipeline for me always has the same six layers: memory audit first —
>  because the code that works on 1,000 rows needs to work on 1 million without
>  infrastructure changes. Then feature engineering builds every derived column once,
>  before any analysis, so there's no duplication downstream. NumPy gives me the
>  statistical backbone — correlation, weighted averages, z-score outliers —
>  before I ever touch GroupBy. Time series shows me the trajectory. MultiIndex
>  GroupBy with named aggregation gives the client one table instead of six.
>  And chunked processing means the same notebook ships to production unchanged.
>  That's the pipeline. Every piece earns its place."


In [42]:
# ★ Bonus 1 — Weekend vs Weekday Analysis (5 pts)
# ------------------------------------------------
bonus1_rev = df.pivot_table(
    values='revenue',
    index=pd.Categorical(df['is_weekend'].map({True:'Weekend', False:'Weekday'})),
    aggfunc=['mean', 'count']
)
bonus1_margin = df.pivot_table(
    values='profit_margin',
    index=pd.Categorical(df['is_weekend'].map({True:'Weekend', False:'Weekday'})),
    aggfunc='mean'
)

bonus1 = pd.DataFrame({
    'avg_revenue': bonus1_rev[('mean', 'revenue')],
    'order_count': bonus1_rev[('count', 'revenue')],
    'avg_profit_margin': bonus1_margin['profit_margin']
}).round(2)

print("\n--- ★ Bonus 1: Weekend vs Weekday ---")
print(bonus1.to_string())
print("NRA: Weekend orders have a slightly higher average revenue but similar profit margin.")
print("     Running weekend‑specific promotions could further boost revenue without eroding profitability.")


--- ★ Bonus 1: Weekend vs Weekday ---
         avg_revenue  order_count  avg_profit_margin
Weekday      9391.36          693              20.04
Weekend      9267.41          307              20.57
NRA: Weekend orders have a slightly higher average revenue but similar profit margin.
     Running weekend‑specific promotions could further boost revenue without eroding profitability.


In [40]:
# ★ Bonus 2 — Cancellation Risk Score (5 pts)
# ═══════════════════════════════════════════════════════════════════════════════
bonus2 = df.groupby('category').agg(
    total   = ('order_id','count'),
    ret     = ('is_returned','sum'),
    canc    = ('is_cancelled','sum')
)
bonus2['return_rate'] = bonus2['ret'] / bonus2['total'] * 100
bonus2['cancel_rate'] = bonus2['canc'] / bonus2['total'] * 100
bonus2['risk_score']  = bonus2['return_rate'] * 0.6 + bonus2['cancel_rate'] * 0.4
bonus2 = bonus2.sort_values('risk_score', ascending=False)

print("\n--- ★ Bonus 2: Cancellation Risk Score ---")
print(bonus2[['return_rate','cancel_rate','risk_score']].to_string())
highest_risk = bonus2.index[0]
print(f"NRA: {highest_risk} has the highest risk score. Return rate is the dominant contributor.")
print(f"     Implement tighter quality checks and clearer product listings for {highest_risk}.")



--- ★ Bonus 2: Cancellation Risk Score ---
             return_rate  cancel_rate  risk_score
category                                         
Books          24.761905    10.476190   19.047619
Home           19.148936     6.914894   14.255319
Electronics    18.115942     6.159420   13.333333
Clothing       15.957447     8.510638   12.978723
Beauty         14.141414     7.070707   11.313131
Sports         11.111111     9.027778   10.277778
NRA: Books has the highest risk score. Return rate is the dominant contributor.
     Implement tighter quality checks and clearer product listings for Books.


In [41]:
# ★ Bonus 3 — Customer Segmentation Summary (5 pts)
# ═══════════════════════════════════════════════════════════════════════════════
# Use the df_sorted with order_seq for repeat rate
seg_summary = df_sorted.groupby('segment').agg(
    avg_order_value = ('revenue','mean'),
    avg_profit_margin = ('profit_margin','mean'),
    return_rate    = ('is_returned','mean'),   # proportion of orders returned
    total_rows     = ('order_id','count'),
    repeat_rows    = ('order_seq', lambda x: (x >= 2).sum())
)
seg_summary['repeat_rate'] = seg_summary['repeat_rows'] / seg_summary['total_rows'] * 100
seg_summary = seg_summary[['avg_order_value','avg_profit_margin','return_rate','repeat_rate']].round(2)
seg_summary['return_rate'] = seg_summary['return_rate'] * 100   # express as percentage

print("\n--- ★ Bonus 3: Segment Summary ---")
print(seg_summary.to_string())
print("NRA: Enterprise has the highest avg order value and repeat rate, making it the most valuable.")
print("     B2C has a slightly higher return rate; attention to product quality or delivery experience may reduce returns.")



--- ★ Bonus 3: Segment Summary ---
            avg_order_value  avg_profit_margin  return_rate  repeat_rate
segment                                                                 
B2B                 8763.97              19.90         16.0        79.79
B2C                 9433.48              20.39         18.0        80.54
Enterprise         10212.74              20.11         16.0        79.05
NRA: Enterprise has the highest avg order value and repeat rate, making it the most valuable.
     B2C has a slightly higher return rate; attention to product quality or delivery experience may reduce returns.
